# Converting NDB trajectories with OpenMiChroM

## Goal

This offline tutorial exercises every converter integrated into `OpenMiChroM.Converters`:

1. NDB → CNDB and CNDB → NDB
2. NDB → PDB and PDB → NDB
3. NDB → SpaceWalk (`.spw`) and SpaceWalk → NDB
4. GROMACS (`.gro`) → NDB
5. Bintu et al. CSV (`model,index,z,x,y`) → NDB

We use a deterministic two-frame, four-bead fixture and assert frame counts, bead counts, coordinates, type mappings, loops, and selected genomic metadata. Files are created in a temporary directory, so the notebook is safe to rerun and needs no network access.

### Provenance

The route inventory and historical format behavior were informed by [mellofariam/NDB-Converters](https://github.com/mellofariam/NDB-Converters), developed by Matheus Mello, Vinícius Contessoto, and Antonio B. Oliveira Junior. That repository does not declare a software license. OpenMiChroM therefore provides a new maintained library implementation; no source files from that repository were copied into the package.

## Setup

Run this notebook from a clone of the integration branch after installing the checkout in editable mode:

```bash
conda env create -f environment.yml
conda activate openmichrom-cndbtools-integration-py310
python -m pip install -e .
python -m jupyter lab
```

The notebook uses only OpenMiChroM's required dependencies. `TemporaryDirectory` prevents tutorial outputs from being mixed with research data.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import warnings

import h5py
import numpy as np

from OpenMiChroM.CndbTools import CndbTools
from OpenMiChroM.Converters import (
    ConverterWarning,
    convert,
    cndb_to_ndb,
    csv_to_ndb,
    gro_to_ndb,
    ndb_to_cndb,
    ndb_to_pdb,
    ndb_to_spw,
    pdb_to_ndb,
    spw_to_ndb,
)

CONVERTER_METHODS = (
    "convert",
    "ndb_to_cndb",
    "cndb_to_ndb",
    "ndb_to_pdb",
    "pdb_to_ndb",
    "ndb_to_spw",
    "spw_to_ndb",
    "gro_to_ndb",
    "csv_to_ndb",
)
assert all(callable(getattr(CndbTools, method)) for method in CONVERTER_METHODS)
print("Converter API available through both OpenMiChroM.Converters and CndbTools")

In [ ]:
temporary_directory = TemporaryDirectory(prefix="openmichrom-converters-")
workspace = Path(temporary_directory.name)
print(f"Temporary converter workspace: {workspace}")

## Steps

### 1. Build a small NDB trajectory

The source has two complete models and ends with `END` after `ENDMDL` records. The fourth bead uses the historical unknown label `UN`; readers normalize it to the current OpenMiChroM label `NA`. A loop record lets us check sidecar behavior in formats that cannot store loops directly.

In [ ]:
EXPECTED_TYPES = ["A1", "A2", "B1", "NA"]
EXPECTED_COORDINATES = np.asarray(
    [
        [
            [0.0, 0.0, 0.0],
            [1.0, 0.5, -0.25],
            [2.0, 1.0, -0.5],
            [3.0, 1.5, -0.75],
        ],
        [
            [0.2, -0.1, 0.3],
            [1.2, 0.4, 0.05],
            [2.2, 0.9, -0.2],
            [3.2, 1.4, -0.45],
        ],
    ],
    dtype=float,
)

ndb_lines = [
    "HEADER    Deterministic OpenMiChroM converter tutorial fixture",
    "TITLE     Two frames, four beads, one chain",
    "ASMBLY    hg38",
    "SEQCHR 1 C1 4 A1 A2 B1 UN",
]
for frame_id, frame in enumerate(EXPECTED_COORDINATES, start=1):
    ndb_lines.append(f"MODEL {frame_id}")
    for bead_index, (label, xyz) in enumerate(
        zip(["A1", "A2", "B1", "UN"], frame), start=1
    ):
        start = 1 + (bead_index - 1) * 1_000
        end = bead_index * 1_000
        x, y, z = xyz
        ndb_lines.append(
            f"CHROM {bead_index} {label} C1 {bead_index} "
            f"{x:.3f} {y:.3f} {z:.3f} {start} {end} 0.000"
        )
    ndb_lines.extend(["TER 5 NA C1", "ENDMDL"])
ndb_lines.extend(["LOOPS 1 4", "MASTER 4 1 1 1000", "END"])

source_ndb = workspace / "source.ndb"
source_ndb.write_text("\n".join(ndb_lines) + "\n", encoding="utf-8")
assert source_ndb.read_text(encoding="utf-8").count("MODEL") == 2
assert source_ndb.read_text(encoding="utf-8").rstrip().endswith("END")
print(source_ndb.read_text(encoding="utf-8"))

### 2. Define semantic checks

A conversion passes only when its output can be read again and its scientific invariants match. Comparing only file existence would miss dropped last frames, shifted coordinates, or a broken type mapping.

In [ ]:
LEGACY_TYPE_CODES = {0: "A1", 1: "A2", 2: "B1", 3: "B2", 4: "B3", 5: "B4", 6: "NA"}


def normalize_type(value):
    if isinstance(value, bytes):
        value = value.decode("utf-8")
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, (int, np.integer)):
        return LEGACY_TYPE_CODES[int(value)]
    label = str(value).strip().upper()
    return "NA" if label == "UN" else label


def read_cndb(path):
    with CndbTools().load(path) as trajectory:
        coordinates = trajectory.xyz()
        types = [normalize_type(value) for value in trajectory.ChromSeq]
        frame_ids = list(trajectory.frame_ids)
    with h5py.File(path, "r") as handle:
        loops = np.asarray(handle["loops"]) if "loops" in handle else np.empty((0, 2), dtype=int)
    return coordinates, types, frame_ids, loops


def inspect_ndb(path, check_name):
    check_cndb = workspace / f"{check_name}.cndb"
    ndb_to_cndb(path, check_cndb)
    return read_cndb(check_cndb)


def assert_trajectory(actual, expected, *, expected_types):
    coordinates, types, frame_ids, _loops = actual
    assert coordinates.shape == expected.shape
    np.testing.assert_allclose(coordinates, expected, rtol=0.0, atol=1.0e-3)
    assert types == list(expected_types)
    assert len(frame_ids) == expected.shape[0]


route_results = {}

### 3. Convert NDB ↔ CNDB

CNDB is the compact HDF5 trajectory format used by OpenMiChroM. This check catches the historical “last frame missing” failure by requiring both source models. It also verifies legacy `UN` → current `NA` normalization and the loop record.

The generic `convert` dispatcher infers the route from suffixes. The same dispatcher is exposed as `CndbTools.convert` for users already working through CNDBTools.

In [ ]:
cndb_path = ndb_to_cndb(source_ndb, workspace / "source.cndb")
generic_cndb = convert(source_ndb, workspace / "generic.cndb")
static_cndb = CndbTools.convert(source_ndb, workspace / "static.cndb")

for candidate in (cndb_path, generic_cndb, static_cndb):
    converted = read_cndb(candidate)
    assert_trajectory(converted, EXPECTED_COORDINATES, expected_types=EXPECTED_TYPES)
    np.testing.assert_array_equal(converted[3], np.asarray([[1, 4]]))

roundtrip_ndb = cndb_to_ndb(cndb_path, workspace / "cndb_roundtrip.ndb")
roundtrip = inspect_ndb(roundtrip_ndb, "cndb_roundtrip_check")
assert_trajectory(roundtrip, EXPECTED_COORDINATES, expected_types=EXPECTED_TYPES)
assert roundtrip_ndb.read_text(encoding="utf-8").count("ENDMDL") == 2
route_results["NDB ↔ CNDB"] = "pass"
print("NDB ↔ CNDB: 2 frames, 4 beads, types, and loops preserved")

### 4. Convert NDB ↔ PDB

PDB is useful for molecular viewers. OpenMiChroM puts chromatin-type hints in its generated PDB records so this package's own NDB → PDB → NDB round trip preserves the four labels and coordinates. Generic third-party PDB files may not contain those hints, genomic intervals, or loops.

In [ ]:
pdb_path = ndb_to_pdb(source_ndb, workspace / "pdb_export.pdb")
pdb_roundtrip_ndb = pdb_to_ndb(
    pdb_path,
    workspace / "pdb_roundtrip.ndb",
    resolution=1_000,
)
pdb_roundtrip = inspect_ndb(pdb_roundtrip_ndb, "pdb_roundtrip_check")
assert_trajectory(pdb_roundtrip, EXPECTED_COORDINATES, expected_types=EXPECTED_TYPES)
assert pdb_path.read_text(encoding="utf-8").count("MODEL") == 2
assert pdb_path.with_suffix(".loops").read_text(encoding="utf-8").strip() == "1 4"
route_results["NDB ↔ PDB"] = "pass"
print("NDB ↔ PDB: coordinate and OpenMiChroM type hints preserved")

### 5. Convert NDB ↔ SpaceWalk (`.spw`)

SpaceWalk text stores traces, chromosome intervals, and coordinates, but it has no chromatin-type column. The reverse conversion therefore assigns `NA` to every bead. Loops travel in a companion `.loops` file.

Do not confuse **SPW** with **SWB**: `.spw` is the SpaceWalk text exchange format handled here, while `.swb` is an HDF5 trajectory produced by OpenMiChroM's SWB reporter. Renaming a suffix does not convert the data.

In [ ]:
with warnings.catch_warnings(record=True) as spw_warnings:
    warnings.simplefilter("always")
    spw_path = ndb_to_spw(
        source_ndb,
        workspace / "spw_export.spw",
        name="converter-tutorial",
    )
assert len(spw_warnings) == 1
assert issubclass(spw_warnings[0].category, ConverterWarning)
spw_loops = spw_path.with_suffix(".loops")
assert spw_loops.read_text(encoding="utf-8").strip() == "1 4"
assert spw_path.read_text(encoding="utf-8").count("trace") == 2
assert "genome=hg38" in spw_path.read_text(encoding="utf-8")

spw_roundtrip_ndb = spw_to_ndb(
    spw_path,
    workspace / "spw_roundtrip.ndb",
    loops=spw_loops,
)
spw_roundtrip = inspect_ndb(spw_roundtrip_ndb, "spw_roundtrip_check")
assert_trajectory(spw_roundtrip, EXPECTED_COORDINATES, expected_types=["NA"] * 4)
np.testing.assert_array_equal(spw_roundtrip[3], np.asarray([[1, 4]]))
route_results["NDB ↔ SPW"] = "pass (types become NA by design)"
print("NDB ↔ SPW: traces, coordinates, intervals, and loops preserved; types are lossy")

### 6. Convert GRO → NDB

A GRO trajectory contains title, atom-count, fixed-width atom, and box records for each frame. The fixture uses the historical OpenMiChroM atom labels `ZA`, `OA`, `FB`, and `UN`, which map to `A1`, `A2`, `B1`, and `NA`. Genomic intervals are reconstructed from `resolution`.

In [ ]:
GRO_TYPES = ["ZA", "OA", "FB", "UN"]


def gro_frame(title, coordinates):
    lines = [title, str(len(coordinates))]
    for atom_index, (atom_type, xyz) in enumerate(zip(GRO_TYPES, coordinates), start=1):
        x, y, z = xyz
        lines.append(
            f"{atom_index:5d}{'ChrA':<5}{atom_type:>5}{atom_index:5d}"
            f"{x:8.3f}{y:8.3f}{z:8.3f}"
        )
    lines.append(f"{10.0:10.5f}{10.0:10.5f}{10.0:10.5f}")
    return lines


gro_path = workspace / "source.gro"
gro_lines = []
for frame_number, frame in enumerate(EXPECTED_COORDINATES, start=1):
    gro_lines.extend(gro_frame(f"OpenMiChroM frame {frame_number}", frame))
gro_path.write_text("\n".join(gro_lines) + "\n", encoding="utf-8")

gro_ndb = gro_to_ndb(
    gro_path,
    workspace / "gro_import.ndb",
    resolution=1_000,
)
gro_import = inspect_ndb(gro_ndb, "gro_import_check")
assert_trajectory(gro_import, EXPECTED_COORDINATES, expected_types=EXPECTED_TYPES)
assert "C1" in gro_ndb.read_text(encoding="utf-8")
route_results["GRO → NDB"] = "pass"
print("GRO → NDB: 2 frames and legacy type labels mapped correctly")

### 7. Convert Bintu et al. CSV → NDB

This importer intentionally supports the historical Bintu et al. row layout `model,index,z,x,y`; it is not a generic CSV schema detector. CSV has no chromatin-type field, so the NDB result uses `NA`. `chromosome`, `genomic_start`, and `resolution` make the reconstructed genomic intervals explicit.

In [ ]:
csv_path = workspace / "bintu_fixture.csv"
csv_lines = [
    "Deterministic Bintu-style converter fixture",
    "Chromosome 21",
]
for frame_number, frame in enumerate(EXPECTED_COORDINATES, start=1):
    for bead_index, (x, y, z) in enumerate(frame, start=1):
        csv_lines.append(f"{frame_number},{bead_index},{z:.3f},{x:.3f},{y:.3f}")
csv_path.write_text("\n".join(csv_lines) + "\n", encoding="utf-8")

csv_ndb = csv_to_ndb(
    csv_path,
    workspace / "csv_import.ndb",
    chromosome=21,
    genomic_start=18_000_000,
    resolution=1_000,
)
csv_import = inspect_ndb(csv_ndb, "csv_import_check")
assert_trajectory(csv_import, EXPECTED_COORDINATES, expected_types=["NA"] * 4)
csv_text = csv_ndb.read_text(encoding="utf-8")
assert "C21" in csv_text
assert "18000000" in csv_text and "18001000" in csv_text
route_results["Bintu CSV → NDB"] = "pass (types become NA by design)"
print("Bintu CSV → NDB: coordinate order and genomic interval reconstruction verified")

## Checks

Every supported route has now been exercised using files written to disk and read back through the public API. These are semantic checks, not filename-only checks.

In [ ]:
expected_routes = {
    "NDB ↔ CNDB",
    "NDB ↔ PDB",
    "NDB ↔ SPW",
    "GRO → NDB",
    "Bintu CSV → NDB",
}
assert set(route_results) == expected_routes
assert all(result.startswith("pass") for result in route_results.values())

print("All 8 directed converter routes passed:")
print("  NDB → CNDB, CNDB → NDB")
print("  NDB → PDB, PDB → NDB")
print("  NDB → SPW, SPW → NDB")
print("  GRO → NDB")
print("  Bintu CSV → NDB")
for route, result in route_results.items():
    print(f"- {route}: {result}")

## What is and is not lossless

| Route | Coordinates / frames | Types | Genomic intervals | Loops |
|---|---|---|---|---|
| NDB ↔ CNDB | Preserved | Preserved; legacy `UN` normalizes to `NA` | Preserved by OpenMiChroM CNDB metadata | Preserved |
| NDB ↔ generated PDB | Preserved to PDB precision | Preserved through OpenMiChroM hints | Generic PDB cannot represent all NDB metadata | Companion `.loops` on export |
| NDB ↔ SPW | Preserved | **Lost; becomes `NA`** | Preserved | Companion `.loops` |
| GRO → NDB | Preserved | Recognized OpenMiChroM labels map; otherwise provide types or use `NA` | Reconstructed from resolution | Supply separately if needed |
| Bintu CSV → NDB | Preserved with `z,x,y` reorder | **Absent; becomes `NA`** | Reconstructed from requested chromosome/start/resolution | Supply separately if needed |

For scientific data, always validate the first **and** last frame, not just frame 1. Also verify chain boundaries and the exact coordinate units expected by the source format.

## Next steps

Use the explicit route functions when you want route-specific options and `convert` for suffix-based dispatch. For large CNDB analysis after conversion, load the result with `CndbTools` and select only the frames and beads you need. Keep the source file until the converted trajectory has passed the checks above.

The next cell removes the temporary fixture directory. Skip it if you want to inspect files during a manual Jupyter session; they will otherwise be removed when the kernel exits.

In [ ]:
generated_files = sorted(path.name for path in workspace.iterdir())
assert generated_files
print(f"Generated and checked {len(generated_files)} temporary files")
temporary_directory.cleanup()
assert not workspace.exists()
print("Temporary converter workspace cleaned")